In [1]:
infile = "~/projects/Voltage_Data/glove/wikishuf10.txt"
!wc $infile

   10000 3010002 38972029 /Users/yoavfreund/projects/Voltage_Data/glove/wikishuf10.txt


In [3]:
import pandas as pd
# Use regex separator and Python engine for maximum flexibility with bad lines
df = pd.read_csv(infile, header=None, sep=r'\s+', on_bad_lines='skip', engine='python')
df.head()

,0,1,2,3,4,5,6,7,8,9,...,291,292,293,294,295,296,297,298,299,300
0,steelers,-0.180579,-0.425886,-0.575671,-0.055740,0.830057,0.709065,0.016221,-0.365534,0.689626,...,-1.009310,-0.540462,0.323385,-0.371456,-0.062482,0.388162,0.084096,-0.427911,0.145135,0.314865
1,provided,-0.015177,0.375130,0.399406,0.382963,0.185270,0.012572,0.103914,-0.044613,-0.210380,...,0.020759,0.331356,0.514022,-0.054313,-0.161005,-0.100283,0.166354,-0.234262,-0.060326,0.145183
2,2019,0.159078,-0.202463,0.048832,-0.126726,-0.242197,-0.090326,-0.121398,-0.259350,0.271986,...,-0.474033,0.402923,-0.305514,0.114831,0.189959,0.766474,-0.018599,-0.321501,0.415208,-0.116852
3,ga.,-0.262486,0.055257,-0.203598,0.008679,-0.272630,-0.150138,-0.660054,0.269485,0.283914,...,0.283440,-0.006786,-0.264685,-0.254293,0.459916,-0.031608,-0.213312,0.291775,-0.251326,0.034483
4,gospel,-0.396376,0.119346,0.020996,0.527459,0.402824,0.478787,-0.317356,-0.088708,0.032543,...,-0.256071,0.011596,0.164632,-0.953014,0.261790,-0.263064,-0.358291,-0.520554,0.346423,0.319620


In [6]:
# Computing Part of Speech (POS) for words using spaCy
import pandas as pd
import numpy as np

# First, let's see what the data looks like
print("Data shape:", df.shape)
print("First few rows:")
print(df.head())

# Assuming the first column contains the words (typical for GloVe format)
# Let's extract the words
words = df.iloc[:, 0].tolist()
print(f"\nFound {len(words)} words")
print("Sample words:", words[:10])

# Method 1: Using spaCy (recommended - more accurate than NLTK)
try:
    import spacy
    
    # Load English model (install with: python -m spacy download en_core_web_sm)
    nlp = spacy.load("en_core_web_sm")
    
    print("\nMethod 1: Using spaCy (Primary method)")
    
    # Process all words efficiently in batches
    batch_size = 1000
    word_to_pos_spacy = {}
    
    print("Processing words in batches...")
    for i in range(0, len(words), batch_size):
        batch = words[i:i+batch_size]
        # Process batch as a single document for efficiency
        doc = nlp(" ".join(batch))
        
        batch_idx = 0
        for token in doc:
            if token.text in batch and batch_idx < len(batch):
                if token.text == batch[batch_idx]:
                    word_to_pos_spacy[token.text] = token.pos_
                    batch_idx += 1
        
        if (i + batch_size) % 5000 == 0:
            print(f"  Processed {i + batch_size} words...")
    
    print(f"Successfully tagged {len(word_to_pos_spacy)} words with spaCy")
    
    print("Sample spaCy POS tags:")
    sample_words = words[:15]
    for word in sample_words:
        pos = word_to_pos_spacy.get(word, 'UNKNOWN')
        print(f"  {word:12} -> {pos}")
    
    # spaCy POS tag meanings:
    spacy_pos_meanings = {
        'NOUN': 'Noun',
        'VERB': 'Verb', 
        'ADJ': 'Adjective',
        'ADV': 'Adverb',
        'PRON': 'Pronoun',
        'DET': 'Determiner',
        'ADP': 'Adposition (preposition/postposition)',
        'CONJ': 'Conjunction',
        'NUM': 'Number',
        'PART': 'Particle',
        'INTJ': 'Interjection',
        'X': 'Other',
        'SPACE': 'Space',
        'PUNCT': 'Punctuation',
        'SYM': 'Symbol'
    }
    
    print(f"\nPOS tag distribution (spaCy):")
    pos_counts = {}
    for pos in word_to_pos_spacy.values():
        pos_counts[pos] = pos_counts.get(pos, 0) + 1
    
    for pos, count in sorted(pos_counts.items(), key=lambda x: x[1], reverse=True):
        meaning = spacy_pos_meanings.get(pos, "Other")
        print(f"  {pos:8}: {count:6} ({meaning})")

except ImportError:
    print("spaCy not available. Install with: pip install spacy")
    print("Then download model with: python -m spacy download en_core_web_sm")
    word_to_pos_spacy = None
except OSError:
    print("spaCy model not found. Install with: python -m spacy download en_core_web_sm")
    word_to_pos_spacy = None

# Fallback Method: Simple heuristic approach if spaCy fails
def simple_pos_heuristics(word):
    """Simple heuristic POS tagging based on word endings"""
    word = word.lower()
    
    # Common endings for different POS
    if word.endswith(('ing', 'ed')) and len(word) > 4:
        return 'VERB'
    elif word.endswith('ly') and len(word) > 3:
        return 'ADV'
    elif word.endswith(('tion', 'sion', 'ness', 'ment', 'ity', 'er', 'or', 'ism', 'ist')):
        return 'NOUN'
    elif word.endswith(('ful', 'less', 'ous', 'ive', 'al', 'ic', 'able', 'ible')):
        return 'ADJ'
    elif word in ['the', 'a', 'an', 'this', 'that', 'these', 'those']:
        return 'DET'
    elif word in ['and', 'or', 'but', 'so', 'yet']:
        return 'CONJ'
    elif word in ['in', 'on', 'at', 'by', 'for', 'with', 'to', 'from', 'about']:
        return 'ADP'
    elif word in ['very', 'quite', 'rather', 'too', 'so']:
        return 'ADV'
    else:
        return 'NOUN'  # Default to noun

# Create a DataFrame with words and their POS tags
print("\nCreating POS-tagged DataFrame...")
df_with_pos = df.copy()
df_with_pos['word'] = df.iloc[:, 0]

if word_to_pos_spacy is not None:
    # Use spaCy results
    df_with_pos['pos_tag'] = df_with_pos['word'].map(word_to_pos_spacy)
    # Fill missing values with heuristics
    missing_mask = df_with_pos['pos_tag'].isna()
    df_with_pos.loc[missing_mask, 'pos_tag'] = df_with_pos.loc[missing_mask, 'word'].apply(simple_pos_heuristics)
    print(f"Added POS tags using spaCy for {len(word_to_pos_spacy)} words")
    print(f"Used heuristics for {missing_mask.sum()} missing words")
else:
    # Fall back to heuristics
    df_with_pos['pos_tag'] = df_with_pos['word'].apply(simple_pos_heuristics)
    print(f"Added POS tags using heuristics for all {len(df)} words")

print("\nSample of DataFrame with POS tags:")
print(df_with_pos[['word', 'pos_tag']].head(15))

print(f"\nFinal POS tag distribution:")
final_pos_counts = df_with_pos['pos_tag'].value_counts()
for pos, count in final_pos_counts.head(10).items():
    meaning = spacy_pos_meanings.get(pos, "Other")
    print(f"  {pos:8}: {count:6} ({meaning})")

Data shape: (9999, 301)
First few rows:
        0         1         2         3         4         5         6    \
0  steelers -0.180579 -0.425886 -0.575671 -0.055740  0.830057  0.709065   
1  provided -0.015177  0.375130  0.399406  0.382963  0.185270  0.012572   
2      2019  0.159078 -0.202463  0.048832 -0.126726 -0.242197 -0.090326   
3       ga. -0.262486  0.055257 -0.203598  0.008679 -0.272630 -0.150138   
4    gospel -0.396376  0.119346  0.020996  0.527459  0.402824  0.478787   

        7         8         9    ...       291       292       293       294  \
0  0.016221 -0.365534  0.689626  ... -1.009310 -0.540462  0.323385 -0.371456   
1  0.103914 -0.044613 -0.210380  ...  0.020759  0.331356  0.514022 -0.054313   
2 -0.121398 -0.259350  0.271986  ... -0.474033  0.402923 -0.305514  0.114831   
3 -0.660054  0.269485  0.283914  ...  0.283440 -0.006786 -0.264685 -0.254293   
4 -0.317356 -0.088708  0.032543  ... -0.256071  0.011596  0.164632 -0.953014   

        295       296       

In [7]:
# Verify the dataframe structure with POS tags
print("DataFrame with POS tags:")
print(f"Shape: {df_with_pos.shape}")
print(f"Columns: {list(df_with_pos.columns)}")
print("\nFirst 10 rows with word and POS tag:")
print(df_with_pos[['word', 'pos_tag']].head(10))

print("\nPOS tag value counts:")
print(df_with_pos['pos_tag'].value_counts().head(15))

# Test specific words mentioned earlier
test_words = ['cathedral', 'steelers', 'provided', 'gospel', 'uncle']
print(f"\nPOS tags for specific test words:")
for word in test_words:
    word_rows = df_with_pos[df_with_pos['word'] == word]
    if not word_rows.empty:
        pos = word_rows['pos_tag'].iloc[0]
        print(f"  {word:12} -> {pos}")
    else:
        print(f"  {word:12} -> NOT FOUND")

DataFrame with POS tags:
Shape: (9999, 303)
Columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 2

In [8]:
df_with_pos

,0,1,2,3,4,5,6,7,8,9,...,293,294,295,296,297,298,299,300,word,pos_tag
0,steelers,-0.180579,-0.425886,-0.575671,-0.055740,0.830057,0.709065,0.016221,-0.365534,0.689626,...,0.323385,-0.371456,-0.062482,0.388162,0.084096,-0.427911,0.145135,0.314865,steelers,NOUN
1,provided,-0.015177,0.375130,0.399406,0.382963,0.185270,0.012572,0.103914,-0.044613,-0.210380,...,0.514022,-0.054313,-0.161005,-0.100283,0.166354,-0.234262,-0.060326,0.145183,provided,VERB
2,2019,0.159078,-0.202463,0.048832,-0.126726,-0.242197,-0.090326,-0.121398,-0.259350,0.271986,...,-0.305514,0.114831,0.189959,0.766474,-0.018599,-0.321501,0.415208,-0.116852,2019,NUM
3,ga.,-0.262486,0.055257,-0.203598,0.008679,-0.272630,-0.150138,-0.660054,0.269485,0.283914,...,-0.264685,-0.254293,0.459916,-0.031608,-0.213312,0.291775,-0.251326,0.034483,ga.,NOUN
4,gospel,-0.396376,0.119346,0.020996,0.527459,0.402824,0.478787,-0.317356,-0.088708,0.032543,...,0.164632,-0.953014,0.261790,-0.263064,-0.358291,-0.520554,0.346423,0.319620,gospel,NOUN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,graduate,0.186076,0.215013,0.688934,-0.368438,-0.441422,0.163604,-0.264886,0.518319,-0.032260,...,0.292001,-0.171310,0.193204,0.005474,0.014301,0.202036,0.310106,0.024693,graduate,NOUN
9995,eastern,0.129160,-0.463495,0.121020,0.340409,0.050741,-0.336649,0.444270,0.164382,0.544275,...,-0.338500,0.071334,0.477465,0.413756,0.150390,0.651748,0.539124,0.205233,eastern,NOUN
9996,2013,0.103718,-0.076963,0.041578,-0.181205,-0.011359,-0.104143,-0.088296,-0.208881,0.015116,...,-0.215588,0.168833,0.053851,0.526847,0.059040,-0.256953,0.198822,-0.170103,2013,NOUN
9997,yacht,-0.588899,-0.034704,-0.510207,0.088372,-0.193545,0.550380,-0.023637,-0.078727,-0.245833,...,0.485991,0.541584,0.173619,0.061063,0.336260,0.070854,0.385835,0.106770,yacht,NOUN


In [ ]:
# Rename columns to have descriptive names
# Column 0: "word" (the word itself)
# Column 1: "label" (POS tag - standing for Part of Speech)
# Columns 2-301: "d1" through "d300" (the 300 dimensional embeddings)

# Create new column names
new_columns = ['word', 'label'] + [f'd{i}' for i in range(1, 301)]

# Apply to the dataframe with POS tags
df_final = df_with_pos.copy()

# Reorder columns to match the new naming scheme
# First column is already 'word', second should be 'pos_tag' (which we'll rename to 'label')
# Then the embedding dimensions (columns 1-300 from original df)
df_final = df_final[['word', 'pos_tag'] + list(range(1, 301))]

# Rename all columns
df_final.columns = new_columns

print("Renamed columns:")
print(f"Shape: {df_final.shape}")
print(f"Column names: {list(df_final.columns[:10])}...{list(df_final.columns[-5:])}")

print(f"\nFirst 5 rows with new column names:")
print(df_final[['word', 'label', 'd1', 'd2', 'd3', 'd299', 'd300']].head())

print(f"\nDataFrame info:")
print(f"Total columns: {len(df_final.columns)}")
print(f"Word column: '{df_final.columns[0]}'")
print(f"Label column: '{df_final.columns[1]}'")
print(f"First embedding dimension: '{df_final.columns[2]}'")
print(f"Last embedding dimension: '{df_final.columns[-1]}'")

print(f"\nLabel (POS) distribution:")
print(df_final['label'].value_counts().head(10))